# Solving Linear Systems

A linear system is a set of equations in which each unknown appears only multiplied by a number. This notebook solves one with Julia's backslash operator, checks the answer, and looks at what happens when the system has no unique solution.

In [ ]:
Pollis.packages("Plots")

In [ ]:
using LinearAlgebra, Plots

## A three-sector economy

An input-output table records how much each sector buys from the others to produce one unit of output. With technical coefficients $C$ and final demand $d$, the gross output $x$ each sector must produce satisfies $x = Cx + d$, that is

$$(I - C)\,x = d.$$

The numbers below are made up for illustration: agriculture, manufacturing and services.

In [ ]:
C = [0.2 0.1 0.1;
     0.3 0.3 0.2;
     0.1 0.2 0.3]   # column j: inputs needed per unit of output of sector j
d = [100.0, 200, 150]  # final demand
A = I - C
x = A \ d

`A \ d` factorises $A$ and solves two triangular systems. It never computes an inverse. Check the answer by multiplying back: the residual should be zero up to rounding.

In [ ]:
r = d - A * x
(residual_norm = norm(r), relative_residual = norm(r) / (opnorm(A) * norm(x)))

## How sensitive is the answer?

The condition number $\kappa(A)$ bounds how much a relative change in $d$ can grow in $x$. Here it is small, so the solution is robust. Raise demand for services by 1% and see how output changes.

In [ ]:
d2 = copy(d); d2[3] *= 1.01
x2 = A \ d2
(condition_number = cond(A), percent_change_in_output = round.(100 .* (x2 .- x) ./ x, digits=3))

## When there is no unique solution

If one equation is a multiple or a combination of the others, $A$ is singular. When elimination meets an exact zero pivot, Julia stops with a `SingularException` rather than return nonsense.

In [ ]:
S1 = [1.0 2; 2 4]   # second row = 2 * first
try
    S1 \ [1.0, 2]
catch e
    e
end

A matrix that is singular only up to rounding is more dangerous. Below, the third row is exactly twice the second minus the first, but rounding leaves a tiny nonzero pivot, so Julia returns an answer.

In [ ]:
S = [1.0 2 3; 4 5 6; 7 8 9]
xs = S \ [1.0, 2, 3]
(solution = xs, rank = rank(S), determinant = det(S), condition_number = cond(S))

The determinant prints as a tiny number rather than exactly zero, and the solution looks harmless. `rank` and `cond` give the verdict: a condition number near $10^{16}$ means no digit of the solution can be trusted, and here infinitely many vectors solve the system.

## Never use the inverse

`inv(A) * d` gives the same answer here, but costs about three times as much as one solve and is less accurate on hard problems ([Higham (2002)](https://doi.org/10.1137/1.9780898718027)). Use `A \ d`.

## References

[Higham (2002)](https://doi.org/10.1137/1.9780898718027), *Accuracy and Stability of Numerical Algorithms*, 2nd edition, SIAM.